# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mukeshboolani786/flyrank-internship-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [5]:
%pip install -q duckdb pandas huggingface_hub

In [6]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token Loaded:", HF_TOKEN is not None)

Token Loaded: True


In [7]:
from huggingface_hub import snapshot_download

warehouse_path = snapshot_download(
    repo_id="FlyRank/internship-warehouse",
    repo_type="dataset",
    token=HF_TOKEN
)

print(warehouse_path)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 24 files:   0%|          | 0/24 [00:00<?, ?it/s]

/root/.cache/huggingface/hub/datasets--FlyRank--internship-warehouse/snapshots/50cbf7c3909d07be4d1b5906b4d09e882e5acbf2


In [22]:
import duckdb
import os

# Construct the full path to the duckdb file
db_path = os.path.join(warehouse_path, "warehouse.duckdb")

# Check if 'con' already exists and is an open DuckDB connection.
# If so, close it to avoid 'ConnectionException' on re-execution.
if 'con' in globals() and isinstance(globals()['con'], duckdb.DuckDBPyConnection):
    try:
        print("Closing existing DuckDB connection before re-establishing...")
        globals()['con'].close()
        del globals()['con'] # Remove the variable to ensure a fresh start
    except Exception as e:
        print(f"Warning: Could not gracefully close existing connection: {e}")

# Connect to the DuckDB database
con = duckdb.connect(database=db_path, read_only=True)

print("DuckDB connection established successfully.")

# List all tables in the database
tables = con.execute("PRAGMA show_tables;").fetchdf()
display(tables)

Closing existing DuckDB connection before re-establishing...
DuckDB connection established successfully.


,name


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

### Unit of analysis + time window: The Contract

*   **Unit of Analysis:** One row represents the daily performance metrics for a unique content item (`content_id`).
*   **Table(s):** I will be using the `fact_content_daily_performance` table.
*   **Time Window:** A mid-panel month, specifically `month=2026-03`.
*   **Label or Proxy for Prediction/Ranking:** `clicks` as a proxy for content engagement/performance.
*   **Deliberately Excluded:** The `_file_path` column, as it contains metadata about the data source file rather than features of the content itself, and `updated_at` as we are focusing on daily aggregates.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [20]:
# Show a sample of the fact_content_daily_performance table for month=2026-03
# This helps to verify the unit of analysis and identify relevant columns.

# Construct the path to the specific month's data
data_path = os.path.join(warehouse_path, 'fact_content_daily_performance', 'month=2026-03', '*.parquet')

# Query the parquet files directly using DuckDB's glob support
df_sample = con.execute(f"SELECT * FROM '{data_path}' LIMIT 5").fetchdf()
display(df_sample)

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

### Fields: feature / label / context / excluded

Based on the `fact_content_daily_performance` table and the goal of predicting/ranking content performance:

*   **Label:**
    *   `clicks`: This is the primary target for predicting content engagement.

*   **Features:**
    *   `impressions`: The number of times the content was shown. A strong indicator of visibility.
    *   `ctr`: Click-through rate. Provides context on content's appeal relative to impressions.
    *   `content_language`: Language of the content, which can be a categorical feature.
    *   `is_evergreen`: A boolean indicating if the content is evergreen, potentially influencing its long-term performance.
    *   `content_type`: The type of content (e.g., article, video).

*   **Context:**
    *   `date`: The date of the daily performance record. Essential for time-series analysis.
    *   `content_id`: Unique identifier for the content item. Used for joining and tracking.
    *   `created_at`: The creation date of the content, providing age context.
    *   `domain_name`: The domain where the content is hosted.

*   **Excluded:**
    *   `_file_path`: Metadata about the data source file. (Why: Not a feature of the content itself, and not relevant for analysis).
    *   `updated_at`: Last update time. (Why: While useful for understanding data freshness, for a daily performance contract, the `date` column provides the relevant time context for the metrics being observed, and `updated_at` can sometimes be noisy or not directly predictive of content performance).
    *   `page_depth`: (Why: Assuming this refers to something like search result page depth, which is a result of ranking, not a feature available at decision moment for content itself.)

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


### 3. Verify it with queries (grain, counts, missing values, windows)

#### Fact 1: Grain verification (One row represents daily performance for a unique content item)

In [23]:
# Query to check the grain: count distinct combinations of report_date and content_hash_id
# and compare it to the total number of rows.

query_grain = f"""SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT CONCAT(report_date, '-', content_hash_id)) AS distinct_combinations
FROM '{data_path}'
"""
df_grain = con.execute(query_grain).fetchdf()
display(df_grain)

if df_grain['total_rows'].iloc[0] == df_grain['distinct_combinations'].iloc[0]:
    print("Verification successful: Each row represents a unique daily performance for a content item.")
else:
    print("Verification failed: There are duplicate entries for daily content performance.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,distinct_combinations
0,9841378,9841378


Verification successful: Each row represents a unique daily performance for a content item.


#### Fact 2: Slice's row count and date span

In [24]:
# Query to get the total row count and the minimum/maximum report_date for the chosen month

query_counts_dates = f"""SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
FROM '{data_path}'
"""
df_counts_dates = con.execute(query_counts_dates).fetchdf()
display(df_counts_dates)

,row_count,min_date,max_date
0,9841378,2026-03-01,2026-03-31


#### Fact 3: Availability (Filter with IS TRUE for `gsc_data_available`)

In [25]:
# Query to filter for rows where gsc_data_available is TRUE and show the count

query_availability = f"""SELECT
    COUNT(*) AS rows_with_gsc_data
FROM '{data_path}'
WHERE gsc_data_available IS TRUE
"""
df_availability = con.execute(query_availability).fetchdf()
display(df_availability)

,rows_with_gsc_data
0,3611061


## 3.1 Five Features (max)

In [26]:
# Select 5 features and the label from the fact_content_daily_performance table for month=2026-03
# For this step, we'll select features available in the current row's context.

feature_cols = [
    'content_hash_id',
    'report_date',
    'gsc_impressions',
    'gsc_avg_position',
    'ga4_pageviews',
    'sessions_organic',
    'scroll_events'
]

label_col = 'gsc_clicks'

# Construct the path to the specific month's data
data_path_features = os.path.join(warehouse_path, 'fact_content_daily_performance', 'month=2026-03', '*.parquet')

# Query the parquet files directly using DuckDB to create the feature frame
df_features = con.execute(f"SELECT {', '.join(feature_cols + [label_col])} FROM '{data_path_features}'").fetchdf()
display(df_features.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,content_hash_id,report_date,gsc_impressions,gsc_avg_position,ga4_pageviews,sessions_organic,scroll_events,gsc_clicks
0,content_b7e512995f79d5a6,2026-03-01,20,3.350000,<NA>,<NA>,<NA>,0
1,content_05597932fe4da067,2026-03-01,1,0.000000,<NA>,<NA>,<NA>,0
2,content_7a105f548d9c6916,2026-03-01,125,4.928000,<NA>,<NA>,<NA>,1
3,content_905aa32a0230694e,2026-03-01,7,4.000000,<NA>,<NA>,<NA>,0
4,content_a3ea9792f793ec72,2026-03-01,11,2.272727,<NA>,<NA>,<NA>,0


### Knowable at the decision moment because…

*   `gsc_impressions`: knowable at the decision moment because it represents the historical visibility of the content up to the point of prediction.
*   `gsc_avg_position`: knowable at the decision moment because it reflects the historical average search engine ranking of the content prior to the prediction.
*   `ga4_pageviews`: knowable at the decision moment because it represents pageviews recorded for the content up to the current daily aggregate.
*   `sessions_organic`: knowable at the decision moment because it reflects traffic from organic search up to the current daily aggregate.
*   `scroll_events`: knowable at the decision moment because it indicates user engagement with the content up to the current daily aggregate.

## 3.2 The Trap: Leakage Lesson

Here, I will demonstrate the leakage lesson by deliberately adding a label-derived column as a feature, observing its impact on a simplistic 'score' (e.g., correlation with the label), and then removing it to show the honest number.

In [27]:
import numpy as np

# Calculate a baseline correlation without leakage
baseline_corr = df_features['gsc_impressions'].corr(df_features['gsc_clicks'])
print(f"Baseline correlation (impressions vs clicks): {baseline_corr:.4f}")

# --- Deliberate Leakage --- #
# Create a leaked feature: a direct derivative of the label, or even the label itself
# For demonstration, let's add 'clicks' directly as a feature, which is a perfect leak.
# In a real scenario, this could be 'CTR' if 'clicks' is the label, or 'average daily clicks over last 3 days' where the 3 days overlap with the label.

df_leaked = df_features.copy()
df_leaked['leaked_clicks_feature'] = df_leaked['gsc_clicks'] # This is the leakage

# Calculate correlation with the leaked feature
leaked_corr = df_leaked['leaked_clicks_feature'].corr(df_leaked['gsc_clicks'])
print(f"Correlation with leaked feature (leaked_clicks_feature vs clicks): {leaked_corr:.4f}")
print("Notice how the score jumped towards perfect due to data leakage.")

# --- Remove Leakage --- #
# Now, remove the leaked feature to represent the honest situation
df_honest = df_leaked.drop(columns=['leaked_clicks_feature'])
print("\nLeaked feature removed. The honest predictive power would be closer to the baseline.")

# For the assignment, we just need to show the experiment, not necessarily a full model.
# The 'df_features' dataframe above is our honest feature set without the leak.

Baseline correlation (impressions vs clicks): 0.5967
Correlation with leaked feature (leaked_clicks_feature vs clicks): 1.0000
Notice how the score jumped towards perfect due to data leakage.

Leaked feature removed. The honest predictive power would be closer to the baseline.


The experiment above clearly shows how including a label-derived feature (`leaked_clicks_feature` which was a direct copy of `gsc_clicks`) artificially inflates the correlation, pushing it towards a perfect score of 1.0. This demonstrates data leakage: using information that would not genuinely be available at the time of prediction, leading to an overestimation of model performance. Our `df_features` dataframe (created earlier) represents the honest feature set without this leakage.

## 4. Data limits

### One named limitation of your slice:

Our chosen data slice (`fact_content_daily_performance` for `month=2026-03`) is limited to daily aggregated performance metrics. This means we cannot analyze intra-day trends or real-time user interactions, which might be crucial for understanding immediate content appeal or predicting short-term viral spikes. We are also limited to metrics available *within* this table and cannot easily incorporate external factors like real-world events or broader market trends without joining with additional datasets, which this contract specifically excludes for this initial phase.

In [21]:
# Display the schema of the fact_content_daily_performance table for month=2026-03
# This helps confirm the fields listed above.

# Query the parquet files directly using DuckDB's glob support and describe the schema
schema_df = con.execute(f"DESCRIBE SELECT * FROM '{data_path}'").fetchdf()
display(schema_df)

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.